# 04. Feature Selection Experiments
**Crop Yield Prediction and Forecasting Using Agricultural Data and Machine Learning**  
*Author: Aryan Puranik*

This notebook conducts systematic feature selection experiments to eliminate multicollinearity, evaluate feature importance across crops, and benchmark feature subsets on out-of-time validation data.

### Research Objectives
1. **Multicollinearity & VIF Analysis:** Empirically quantify variance inflation across all numerical predictors to confirm the $\text{Fertilizer} \approx N + P + K$ dependency.
2. **Mitigating Crop Scale Dominance:** Diagnose why tree algorithms over-attribute importance to high-yield crops (e.g., Sugarcane) and normalize target scale to reveal genuine agronomic drivers.
3. **Mutual Information (MI) Ranking:** Measure non-linear dependency of agricultural and engineered historical features.
4. **Expanding-Window Cross-Validation:** Compare feature selection strategies (All features, Reduced VIF, MI Top 5, LASSO, RFE) across chronological folds (1966–2011).
5. **Validation Benchmark & Ladder Integration:** Evaluate selected configurations on 2012–2014 out-of-time validation data against the historical naive baseline (314.7 kg/ha) established in Notebook 03.
6. **Feature Handoff:** Export standardized feature subsets for subsequent model-specific notebooks (starting with Linear Regression).


## 1. Setup, Imports, and Data Loading

We load the engineered datasets (`train_engineered.csv`, `val_engineered.csv`) and standardized feature groups (`feature_sets.json`) produced in Notebook 03.


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression, SelectKBest, RFE, SelectFromModel
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "model_evaluation"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train_engineered.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val_engineered.csv")

with open(PROCESSED_DIR / "feature_sets.json", "r") as f:
    feature_sets = json.load(f)

print(f"Training set:   {train_df.shape[0]:,} rows, {train_df.shape[1]} columns (Years: {train_df['Year'].min()}-{train_df['Year'].max()})")
print(f"Validation set: {val_df.shape[0]:,} rows, {val_df.shape[1]} columns (Years: {val_df['Year'].min()}-{val_df['Year'].max()})")
print("\nLoaded Feature Sets:", list(feature_sets.keys()))


Training set:   176,194 rows, 30 columns (Years: 1966-2011)
Validation set: 11,345 rows, 30 columns (Years: 2012-2014)

Loaded Feature Sets: ['E0_climate', 'E1_climate_soil', 'E2_traditional_ag', 'E3_engineered_full']


## 2. Multicollinearity & Variance Inflation Factor (VIF) Analysis

In agricultural datasets, `Fertilizer` is typically recorded as total NPK application. We test VIF on all 10 raw numerical predictors to verify multicollinearity.
A VIF exceeding 5 to 10 indicates severe collinearity that distorts linear model coefficients.


In [4]:
num_all_10 = [
    "Year", "Temperature", "Rainfall", "Humidity",
    "pH", "Nitrogen", "Phosphorus", "Potassium",
    "Fertilizer", "Pesticide"
]

def calculate_vif(df, features):
    X = df[features].dropna().values
    vif_dict = {}
    for i, col in enumerate(features):
        other = np.delete(X, i, axis=1)
        target = X[:, i]
        r2 = LinearRegression().fit(other, target).score(other, target)
        vif = 1.0 / max(1.0 - r2, 1e-12)
        vif_dict[col] = vif
    return pd.Series(vif_dict, name="VIF")

# 1. Full 10-feature set (with Fertilizer)
vif_with_fert = calculate_vif(train_df, num_all_10)

# 2. Reduced set (excluding Fertilizer)
num_without_fert = [c for c in num_all_10 if c != "Fertilizer"]
vif_without_fert = calculate_vif(train_df, num_without_fert)

vif_comparison = pd.DataFrame({
    "VIF (With Fertilizer)": vif_with_fert,
    "VIF (Without Fertilizer)": vif_without_fert
})

display(vif_comparison.round(2))

print(f"\nConclusion: 'Fertilizer' exhibits extreme collinearity (VIF > {vif_with_fert['Fertilizer']:.0f}) because Fertilizer ≈ N + P + K.")
print("Dropping 'Fertilizer' brings all remaining numerical predictors below VIF = 6.0.")


             VIF (With Fertilizer)  VIF (Without Fertilizer)
Fertilizer            1.153316e+08                       NaN
Humidity              1.700000e+00                      1.70
Nitrogen              4.914896e+07                      3.90
Pesticide             4.200000e+00                      4.20
Phosphorus            7.747410e+06                      5.77
Potassium             3.670863e+06                      2.54
Rainfall              1.430000e+00                      1.43
Temperature           1.130000e+00                      1.13
Year                  4.710000e+00                      4.71
pH                    1.130000e+00                      1.13

Conclusion: 'Fertilizer' exhibits extreme collinearity (VIF > 115331608) because Fertilizer ≈ N + P + K.
Dropping 'Fertilizer' brings all remaining numerical predictors below VIF = 6.0.


## 3. Tree-Based Feature Importance & Mitigating Crop Scale Dominance

When Random Forest is fitted directly on raw `Yield (kg/ha)`, `Crop_Sugarcane` absorbs ~60% of total Gini impurity importance because Sugarcane yield (~10,000 kg/ha) is an order of magnitude higher than cereals (~2,000 kg/ha) or pulses (~600 kg/ha).

To reveal what features drive **within-crop yield variations**, we also evaluate feature importance on **per-crop standardized yield**:
$$z = \frac{Y - \mu_{\text{crop}}}{\sigma_{\text{crop}}}$$


In [6]:
# Sample 20,000 rows for fast, reproducible tree importance
sample_train = train_df.sample(n=min(20000, len(train_df)), random_state=RANDOM_SEED)

num_features_reduced = ["Year", "Temperature", "Rainfall", "Humidity", "pH", "Nitrogen", "Phosphorus", "Potassium", "Pesticide"]
cat_features_reduced = ["Region", "Crop", "Soil Type"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features_reduced),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features_reduced)
])

X_sample = preprocessor.fit_transform(sample_train)
cat_names = list(preprocessor.named_transformers_["cat"].get_feature_names_out(cat_features_reduced))
all_feature_names = num_features_reduced + cat_names

# 1. Raw Yield Model
rf_raw = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1)
rf_raw.fit(X_sample, sample_train["Yield"])
importances_raw = pd.Series(rf_raw.feature_importances_, index=all_feature_names).sort_values(ascending=False)

# 2. Per-Crop Standardized Model
crop_means = sample_train.groupby("Crop")["Yield"].transform("mean")
crop_stds = sample_train.groupby("Crop")["Yield"].transform("std").replace(0, 1)
y_standardized = (sample_train["Yield"] - crop_means) / crop_stds

rf_std = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1)
rf_std.fit(X_sample, y_standardized)
importances_std = pd.Series(rf_std.feature_importances_, index=all_feature_names).sort_values(ascending=False)

top_comparison = pd.DataFrame({
    "Raw Yield Top 10 Features": importances_raw.head(10).index,
    "Raw Importance": importances_raw.head(10).values.round(4),
    "Standardized Top 10 Features": importances_std.head(10).index,
    "Standardized Importance": importances_std.head(10).values.round(4)
})
display(top_comparison)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
importances_raw.head(8).plot(kind="barh", ax=axes[0], color="crimson").invert_yaxis()
axes[0].set_title("Feature Importance: Raw Yield\n(Dominated by Sugarcane Scale)")
axes[0].set_xlabel("Gini Importance")

importances_std.head(8).plot(kind="barh", ax=axes[1], color="teal").invert_yaxis()
axes[1].set_title("Feature Importance: Per-Crop Standardized\n(Uncovers Agronomic Drivers: N, P, Rain, Humidity)")
axes[1].set_xlabel("Gini Importance")
plt.tight_layout()
plt.show()


  Raw Yield Top 10 Features  ...  Standardized Importance
0            Crop_Sugarcane  ...                   0.2980
1                 Potassium  ...                   0.0617
2                  Nitrogen  ...                   0.0561
3                  Humidity  ...                   0.0498
4     Soil Type_Inceptisols  ...                   0.0474
5                Crop_Wheat  ...                   0.0385
6                 Crop_Rice  ...                   0.0338
7               Temperature  ...                   0.0303
8                Crop_Maize  ...                   0.0299
9                Phosphorus  ...                   0.0242

[10 rows x 4 columns]


## 4. Mutual Information (MI) Ranking

Mutual Information measures both linear and non-linear associations with `Yield`. We compute MI on a representative training sample across traditional agronomic features and engineered historical/anomaly features.


In [8]:
mi_features = [
    "Temperature", "Rainfall", "Humidity", "pH", "Nitrogen", "Phosphorus", "Potassium", "Pesticide",
    "Temperature_Anomaly", "Rainfall_Anomaly", "Yield_Lag1", "Yield_Rolling3", "Yield_Change_1to5"
]

mi_sample = train_df.sample(n=min(20000, len(train_df)), random_state=RANDOM_SEED)
X_mi = mi_sample[mi_features].fillna(mi_sample[mi_features].median())
y_mi = mi_sample["Yield"]

mi_scores = mutual_info_regression(X_mi, y_mi, random_state=RANDOM_SEED)
mi_series = pd.Series(mi_scores, index=mi_features).sort_values(ascending=False)

print("Mutual Information Scores with Yield:")
display(pd.DataFrame({"Mutual Information": mi_series.round(4)}))

fig, ax = plt.subplots(figsize=(8, 4))
mi_series.plot(kind="bar", ax=ax, color="darkslateblue")
ax.set_title("Mutual Information Ranking (Agronomic vs. Historical Features)")
ax.set_ylabel("MI Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


Mutual Information Scores with Yield:
                     Mutual Information
Yield_Lag1                       1.0444
Yield_Rolling3                   0.9964
Yield_Change_1to5                0.2046
pH                               0.0686
Nitrogen                         0.0444
Humidity                         0.0407
Phosphorus                       0.0361
Temperature_Anomaly              0.0340
Pesticide                        0.0302
Temperature                      0.0302
Potassium                        0.0237
Rainfall                         0.0138
Rainfall_Anomaly                 0.0107


## 5. Chronological Expanding-Window Cross-Validation

We evaluate feature selection algorithms across three temporal expanding-window folds:
- **Fold 1:** Train $\le 1999$, Validate 2000–2003
- **Fold 2:** Train $\le 2003$, Validate 2004–2007
- **Fold 3:** Train $\le 2007$, Validate 2008–2011

Strategies tested with Ridge Regression:
1. **All Features Baseline** (All 10 numerical + 4 categorical)
2. **Reduced VIF Baseline** (Excluding redundant Season & Fertilizer)
3. **MI Top 5** (Top 5 numerical features by fold-level Mutual Information)
4. **LASSO Selection** (L1 penalty-based feature filtering)
5. **RFE Top 5** (Recursive Feature Elimination)


In [10]:
cv_folds = [
    (1999, 2000, 2003),
    (2003, 2004, 2007),
    (2007, 2008, 2011)
]

cat_base = ["Region", "Crop", "Season", "Soil Type"]
num_base = ["Year", "Temperature", "Rainfall", "Humidity", "pH", "Nitrogen", "Phosphorus", "Potassium", "Fertilizer", "Pesticide"]

cat_reduced = ["Region", "Crop", "Soil Type"]
num_reduced = ["Year", "Temperature", "Rainfall", "Humidity", "pH", "Nitrogen", "Phosphorus", "Potassium", "Pesticide"]

def build_pipeline(num_cols, cat_cols, method="none"):
    num_steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    if method == "MI":
        def mi_scorer(X, y):
            idx = np.random.RandomState(RANDOM_SEED).choice(len(y), size=min(15000, len(y)), replace=False)
            return mutual_info_regression(X[idx], np.asarray(y)[idx], random_state=RANDOM_SEED)
        num_steps.append(("selector", SelectKBest(score_func=mi_scorer, k=min(5, len(num_cols)))))
    elif method == "LASSO":
        num_steps.append(("selector", SelectFromModel(Lasso(alpha=5.0, max_iter=5000), threshold="median")))
    elif method == "RFE":
        num_steps.append(("selector", RFE(estimator=Ridge(alpha=1.0), n_features_to_select=min(5, len(num_cols)))))

    pre = ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ])
    return Pipeline([("pre", pre), ("reg", Ridge(alpha=1.0, solver="lsqr"))])

experiments = {
    "All Features Baseline": (num_base, cat_base, "none"),
    "Reduced VIF Baseline": (num_reduced, cat_reduced, "none"),
    "MI Top 5": (num_reduced, cat_reduced, "MI"),
    "LASSO Selection": (num_reduced, cat_reduced, "LASSO"),
    "RFE Top 5": (num_reduced, cat_reduced, "RFE")
}

cv_records = []
for exp_name, (nums, cats, method) in experiments.items():
    for train_end, val_start, val_end in cv_folds:
        f_train = train_df[train_df["Year"] <= train_end]
        f_val = train_df[train_df["Year"].between(val_start, val_end)]
        cols = nums + cats
        model = build_pipeline(nums, cats, method)
        model.fit(f_train[cols], f_train["Yield"])
        preds = model.predict(f_val[cols])
        
        mae = mean_absolute_error(f_val["Yield"], preds)
        rmse = np.sqrt(mean_squared_error(f_val["Yield"], preds))
        r2 = r2_score(f_val["Yield"], preds)
        cv_records.append({"Method": exp_name, "Fold": f"{val_start}-{val_end}", "MAE": mae, "RMSE": rmse, "R2": r2})

cv_df = pd.DataFrame(cv_records)
cv_summary = cv_df.groupby("Method")[["MAE", "RMSE", "R2"]].mean().sort_values("MAE")
print("Expanding-Window Cross-Validation Summary (1966-2011):")
display(cv_summary.round(2))


Expanding-Window Cross-Validation Summary (1966-2011):
                          MAE    RMSE    R2
Method                                     
All Features Baseline  540.30  871.67  0.67
Reduced VIF Baseline   540.49  871.70  0.67
RFE Top 5              540.73  871.39  0.67
LASSO Selection        549.14  875.62  0.66
MI Top 5               551.61  877.33  0.66


## 6. Official Validation Period Benchmark (2012–2014) & Historical Handoff

Now, we evaluate the feature configurations on the **unseen 2012–2014 Validation Period** (trained on all data $\le 2011$).
Crucially, we evaluate:
1. **Traditional Features Baseline** (No historical lags, MAE ~628 kg/ha).
2. **Reduced VIF Baseline** (No Fertilizer, No Season).
3. **Engineered Feature Set $E_3$** (Traditional features + Historical yield lags & anomaly indicators).

We compare these directly against the **Rolling 3-Year Mean Heuristic (314.7 kg/ha)** established in Notebook 03.


In [12]:
# Define validation cohort (full and complete history)
val_complete_mask = val_df["History_3yr_Complete"] == 1
val_complete = val_df[val_complete_mask]

# Feature definitions for validation comparison
hist_features = [
    "Yield_Lag1", "Yield_Rolling3", "Yield_Rolling3_Std", "Yield_Change_1to5",
    "History_Lag1_Available", "History_3yr_Complete",
    "Temperature_Anomaly", "Rainfall_Anomaly"
]

eval_configs = {
    "Ridge: Traditional Baseline (No Lags)": (num_base, cat_base),
    "Ridge: Reduced VIF (No Lags)": (num_reduced, cat_reduced),
    "Ridge: Feature Set E3 (With Historical Lags)": (num_reduced + hist_features, cat_reduced)
}

val_eval_results = []

# Baseline benchmarks from Notebook 03 for reference
val_eval_results.append({
    "Model Configuration": "Naive Benchmark: Previous Year (Lag 1)",
    "Features": "Lag1 only",
    "Cohort": "Complete History (N=9,966)",
    "MAE (kg/ha)": 328.35,
    "RMSE (kg/ha)": 593.71,
    "R2": 0.8823,
    "MAP (%)": 29.90
})

val_eval_results.append({
    "Model Configuration": "Naive Benchmark: Rolling 3-Year Mean",
    "Features": "Rolling3 only",
    "Cohort": "Complete History (N=9,966)",
    "MAE (kg/ha)": 314.66,
    "RMSE (kg/ha)": 562.34,
    "R2": 0.8944,
    "MAP (%)": 29.39
})

for name, (nums, cats) in eval_configs.items():
    cols = nums + cats
    pipe = Pipeline([
        ("pre", ColumnTransformer([
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), nums),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cats)
        ])),
        ("reg", Ridge(alpha=1.0))
    ])
    pipe.fit(train_df[cols], train_df["Yield"])
    
    # 1. Evaluate on Complete History Cohort
    preds_comp = pipe.predict(val_complete[cols])
    y_comp = val_complete["Yield"].values
    err_comp = y_comp - preds_comp
    mae_c = mean_absolute_error(y_comp, preds_comp)
    rmse_c = np.sqrt(mean_squared_error(y_comp, preds_comp))
    r2_c = r2_score(y_comp, preds_comp)
    valid_c_map = y_comp >= 10.0
    map_c = 100.0 * np.mean(np.abs(err_comp[valid_c_map] / y_comp[valid_c_map]))
    
    val_eval_results.append({
        "Model Configuration": name,
        "Features": f"{len(nums)} num, {len(cats)} cat",
        "Cohort": "Complete History (N=9,966)",
        "MAE (kg/ha)": mae_c,
        "RMSE (kg/ha)": rmse_c,
        "R2": r2_c,
        "MAP (%)": map_c
    })
    
    # 2. Evaluate on Full Validation Cohort
    preds_full = pipe.predict(val_df[cols])
    y_full = val_df["Yield"].values
    err_full = y_full - preds_full
    mae_f = mean_absolute_error(y_full, preds_full)
    rmse_f = np.sqrt(mean_squared_error(y_full, preds_full))
    r2_f = r2_score(y_full, preds_full)
    valid_f_map = y_full >= 10.0
    map_f = 100.0 * np.mean(np.abs(err_full[valid_f_map] / y_full[valid_f_map]))
    
    val_eval_results.append({
        "Model Configuration": name,
        "Features": f"{len(nums)} num, {len(cats)} cat",
        "Cohort": "Full Validation (N=11,345)",
        "MAE (kg/ha)": mae_f,
        "RMSE (kg/ha)": rmse_f,
        "R2": r2_f,
        "MAP (%)": map_f
    })

val_summary_df = pd.DataFrame(val_eval_results)
print("=== OFFICIAL 2012-2014 VALIDATION BENCHMARK RESULTS ===")
display(val_summary_df.round(2))

# Save summary report for downstream use
val_summary_df.to_csv(REPORTS_DIR / "feature_selection_validation_benchmark.csv", index=False)


=== OFFICIAL 2012-2014 VALIDATION BENCHMARK RESULTS ===
                            Model Configuration  ...  MAP (%)
0        Naive Benchmark: Previous Year (Lag 1)  ...    29.90
1          Naive Benchmark: Rolling 3-Year Mean  ...    29.39
2         Ridge: Traditional Baseline (No Lags)  ...    73.11
3         Ridge: Traditional Baseline (No Lags)  ...    73.02
4                  Ridge: Reduced VIF (No Lags)  ...    73.11
5                  Ridge: Reduced VIF (No Lags)  ...    73.02
6  Ridge: Feature Set E3 (With Historical Lags)  ...    33.47
7  Ridge: Feature Set E3 (With Historical Lags)  ...    35.64

[8 rows x 7 columns]


## 7. Selected Feature Set & Downstream Handoff

### Decision Summary
1. **Drop Redundant Features:** `Season` (exact proxy of Crop) and `Fertilizer` (VIF > $10^8$ proxy of N+P+K) are dropped.
2. **Include Historical Lags:** Integrating `Yield_Lag1`, `Yield_Rolling3`, and anomaly signals reduces Ridge MAE from **628.86 kg/ha** to **319.80 kg/ha** (a ~49% error reduction), closely matching the strong rolling 3-year mean baseline (314.66 kg/ha).
3. **Selected Final Feature Set for Subsequent Model Notebooks:**
   - **Numerical:** `['Year', 'Temperature', 'Rainfall', 'Humidity', 'pH', 'Nitrogen', 'Phosphorus', 'Potassium', 'Pesticide', 'Yield_Lag1', 'Yield_Rolling3', 'Yield_Rolling3_Std', 'Yield_Change_1to5', 'History_Lag1_Available', 'History_3yr_Complete', 'Temperature_Anomaly', 'Rainfall_Anomaly']`
   - **Categorical:** `['Region', 'Crop', 'Soil Type']`
   - **Target:** `Yield` (measured in kg/ha)

We export this configuration into `selected_features.json` to be consumed by each sequential model notebook (starting with **Linear Regression**).


In [14]:
selected_configuration = {
    "numerical_features": num_reduced + hist_features,
    "categorical_features": cat_reduced,
    "target": "Yield",
    "description": "Selected feature set combining non-collinear agronomic variables, weather anomalies, and historical yield lags."
}

with open(PROCESSED_DIR / "selected_features.json", "w") as f:
    json.dump(selected_configuration, f, indent=2)

print("Saved selected feature configuration to data/processed/selected_features.json")
print(f"Total features selected: {len(selected_configuration['numerical_features']) + len(selected_configuration['categorical_features'])}")
print("Downstream modeling notebooks can now proceed sequentially starting with Linear Regression.")


Saved selected feature configuration to data/processed/selected_features.json
Total features selected: 20
Downstream modeling notebooks can now proceed sequentially starting with Linear Regression.
